In [4]:
!pip install lmstudio wikipedia python-docx

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/4 [wsproto]
   ---------- ----------------------------- 1/4 [msgspec]
   ---------- ----------------------------- 1/4 [msgspec]
   -------------------- ------------------- 2/4 [httpx-ws]
   ------------------------------ --------- 3/4 [lmstudio]
   ------------------------------ --------- 3/4 [lmstudio]
   ------------------------------ --------- 3/4 [lmstudio]
   ------------------------------ --------- 3/4 [lmstudio]
   ------------------------------ --------- 3/4 [lmstudio]
   ---------------------------------------- 4/4 [lmstudio]




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\Jios Matthew Laluyan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [53]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

MODEL = "hermes-3-llama-3.1-8b"

In [54]:
chat_history = [
    {
        "role": "system",
        "content": "You are a concise assistant. Answer clearly and briefly."
    }
]

In [55]:
def calculator(expr):
    try:
        return f"Result: {eval(expr)}"
    except:
        return "Invalid expression"

In [56]:
import re
import wikipedia

def clean_query(text):
    text = text.lower()
    text = re.sub(r"\b(wiki|search|definition|of|the|what is|explain)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def wiki_search(user_input):
    query = clean_query(user_input)

    if not query:
        return "User: " + user_input + "\n\nAssistant:\nInvalid query"

    try:
        summary = wikipedia.summary(query, sentences=5)

        return "User: " + user_input + "\n\nAssistant:\n" + summary

    except:
        return "User: " + user_input + "\n\nAssistant:\nNo result found"

In [66]:
from docx import Document

def save_to_doc(mode="all", keyword=None):
    doc = Document()
    doc.add_heading("Chat Export", 0)

    for msg in chat_history:
        role = msg["role"]
        content = msg["content"]

        # FULL CHAT
        if mode == "all":
            doc.add_paragraph(role.upper() + ":")
            doc.add_paragraph(content)
            doc.add_paragraph("")

        # FILTERED CHAT
        elif mode == "filter" and keyword:
            if keyword.lower() in content.lower():
                doc.add_paragraph(role.upper() + ":")
                doc.add_paragraph(content)
                doc.add_paragraph("")

    filename = "chat_export.docx"
    doc.save(filename)

    return "Saved: " + filename

In [58]:
def ask_ai(user_input):
    chat_history.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model=MODEL,
        messages=chat_history
    )

    reply = response.choices[0].message.content

    chat_history.append({"role": "assistant", "content": reply})

    return reply

In [ ]:
import re

def controller(user_input):
    text = user_input.lower()

    # SAVE FULL CHAT
    if "save" in text and "all" in text:
        return save_to_doc(mode="all")

    # SAVE FILTERED CHAT (user chooses topic)
    if "save" in text and "about" in text:
        keyword = text.split("about")[-1].strip()
        return save_to_doc(mode="filter", keyword=keyword)

    # SIMPLE SAVE COMMAND
    if "save" in text and "doc" in text:
        return save_to_doc(mode="all")

    # CALCULATOR
    if "calculate" in text:
        expr = user_input.replace("calculate", "")
        return calculator(expr.strip())

    # WIKI
    if "wiki" in text or "definition" in text or "what is" in text:
        return wiki_search(user_input)

    # DOCUMENT (manual)
    if "document" in text:
        return create_doc("Generated Document", user_input)

    # CHAT
    return ask_ai(user_input)

In [65]:
import textwrap

def print_wrapped(text, width=80):
    print(textwrap.fill(text, width=width))


while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    response = controller(user_input)

    print("\nUser:", user_input)
    print("Assistant:")
    print_wrapped(response)
    print("\n" + "-"*50 + "\n")


User: Hello, I'm Matthew a 3rd yr Computer Engineering Student
Assistant:
You have already introduced yourself as Matthew, a third-year Computer
Engineering student. If you would like me to provide further assistance related
to your course or studies, please let me know how I can help.

--------------------------------------------------


User: save it sa docs
Assistant:
To save this conversation as a document, please follow these steps:  1. Open a
new text file in any text editor or word processor (like Microsoft Word,
LibreOffice Writer, or even a simple Notepad). 2. Copy the entire conversation
from your chat history. 3. Paste the copied text into your new text file. 4.
Save the file with an appropriate name and file format (e.g., .docx for
Microsoft Word, .txt for plain text).  Remember to back up the document in
multiple locations for safety.

--------------------------------------------------

